# 第7回：線形分類入門 — タイタニックデータで学ぶ分類モデル

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/DS_Seminar1_Public/blob/main/notebooks/session07_linear_classification.ipynb)

**DSゼミナールⅠ（2026年度）**  
熊本大学 データサイエンス学科

前回（第5・6回）ではタイタニック号の乗客データを使い、機械学習（分類）の基本概念と8種類の分類モデルを学びました。今回はその中から**「単純な閾値」「決定木」「Logistic回帰」**の3つに焦点を当て、モデルが「どうやって予測しているか」を深く理解します。

---

## 📋 本日の目標

1. タイタニックデータの読み込み・前処理を復習できる（欠損値補完・カテゴリ変数変換・データ分割）
2. 単純な閾値ベースの分類がどのように機能するか理解できる
3. 決定木の仕組みと可視化ができる
4. Logistic回帰の原理とscikit-learnでの実装ができる
5. 3つのモデルを比較し、それぞれの良し悪しを考察できる

---

## 0. 準備：ライブラリの読み込み


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import subprocess

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

sns.set_theme(style="whitegrid")

# Google Colab で matplotlib の日本語が文字化けしないようにする
try:
    import japanize_matplotlib
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "japanize-matplotlib", "-q"])
    import japanize_matplotlib

plt.rcParams["font.family"] = "IPAexGothic"
plt.rcParams["axes.unicode_minus"] = False

print("ライブラリの読み込み完了！")


---

## 1. タイタニックデータの読み込みと前処理（復習）

前回の処理を一通り復習します。データは scikit-learn が提供する `fetch_openml` から取得します。

### 1-1. データの読み込み


In [ ]:
from sklearn.datasets import fetch_openml

titanic = fetch_openml(name="titanic", version=1, as_frame=True, parser="auto")
df = titanic.frame
print(f"データサイズ: {df.shape}")
print(df.head(3))


### 1-2. 欠損値の確認


In [ ]:
print("=== 欠損値の数 ===")
print(df.isnull().sum())
print(f"\n総欠損数: {df.isnull().sum().sum()}")


> [!NOTE]
> OpenML 版のタイタニックデータ（1309行・14列）では、主に `age`（年齢）に263件、`fare`（運賃）に1件、`cabin`（客室）に1014件、`embarked`（出港地）に2件、`boat`（救命ボート）に823件、`body`（遺体番号）に1188件、`home.dest`（自宅/目的地）に564件の欠損があります。総欠損数は3855件です。

### 1-3. 今回の前処理

以下の一連の処理を実行してください。


In [ ]:
# ① 使う列だけ選択
cols = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']
df_ml = df[cols].copy()

# ② 欠損値の処理（age の欠損を中央値で埋める）
print("処理前の欠損値:")
print(df_ml.isnull().sum())

df_ml['age'] = df_ml['age'].fillna(df_ml['age'].median())
df_ml['fare'] = df_ml['fare'].fillna(df_ml['fare'].median())

print("\n処理後の欠損値:")
print(df_ml.isnull().sum())


In [ ]:
# ③ 文字列を数値に変換（sex: male → 0, female → 1）
# 機械学習モデルは文字列をそのまま扱えないため数値に変換する
df_ml['sex'] = df_ml['sex'].map({'male': 0, 'female': 1})

print("変換後の先頭5行:")
df_ml.head()


In [ ]:
# ④ 説明変数(X)と目的変数(y)に分ける
X = df_ml.drop('survived', axis=1)
y = df_ml['survived']

print("説明変数 X の形状:", X.shape)
print("目的変数 y の形状:", y.shape)
print("\n説明変数の列:", X.columns.tolist())
print("目的変数の値の種類:", y.unique())


### 1-4. 訓練データ・テストデータに分割


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"訓練データ: {X_train.shape[0]} 件")
print(f"テストデータ: {X_test.shape[0]} 件")


---

### 📝 演習 1-1（復習）

上記の前処理コードをコピーして実行し、出力を確認してください。以下の問いに答えてください。

1. `age` の中央値は何でしたか？
2. 欠損値処理の前後で、`age` の欠損数はどう変わりましたか？
3. 訓練データとテストデータの分割比率は何対何ですか？
4. `stratify=y` を指定する理由は何ですか？

---

## 2. 単純な閾値による分類

最もシンプルな分類方法です。「ある特徴量の値が、ある基準値より大きいか小さいか」だけで2クラスに分類します。この基準値のことを**閾値（threshold）**と呼びます。

この章のポイントは、**閾値を少し変えるだけで、正解率が良くなったり悪くなったりする**ことを目で確認することです。

### 2-1. 概念

単純な閾値分類は、以下のように定義されます。

```
if 特徴量 < 閾値:
    予測 = 生存(1)
else:
    予測 = 死亡(0)
```

例えば「年齢が 30 歳未満なら生存、30歳以上なら死亡」といったルールです。これはとても単純ですが、分類モデルの基本的な考え方を理解するには便利です。

ただし、閾値分類には重要な弱点があります。

- 1つの特徴量しか見ていない
- 閾値をどこに置くかで結果が大きく変わる
- データの分布によっては、どの閾値を選んでもあまり良い分類にならない

まずは、年齢と生存/死亡の関係を図で確認してみます。


In [ ]:
age_plot_df = pd.concat([X_test, y_test], axis=1).copy()
age_plot_df["survived_label"] = age_plot_df["survived"].map({0: "死亡(0)", 1: "生存(1)"})

plt.figure(figsize=(8, 5))
sns.histplot(
    data=age_plot_df,
    x="age",
    hue="survived_label",
    bins=20,
    multiple="stack",
    palette={"死亡(0)": "tab:blue", "生存(1)": "tab:orange"}
)
plt.xlabel("年齢", fontsize=12)
plt.ylabel("人数", fontsize=12)
plt.title("年齢分布と生存/死亡", fontsize=13)
plt.tight_layout()
plt.show()


このグラフでは、横軸が年齢、色が生存/死亡を表します。もし「若い人はほとんど生存、年齢が高い人はほとんど死亡」のようにはっきり分かれていれば、年齢だけでも高い正解率が期待できます。しかし実際には、生存者と死亡者がかなり重なっています。この「重なり」が、分類を難しくしています。

### 2-2. 年齢を特徴量とした閾値分類

年齢を軸に、様々な閾値で分類してみましょう。ここでは、

```
年齢 < 閾値 → 生存(1)
年齢 >= 閾値 → 死亡(0)
```

というルールを使います。


In [ ]:
# 閾値を 10 から 60 まで 5刻みで試す
thresholds = list(range(10, 65, 5))
accuracies = []

for thresh in thresholds:
    # 年齢が閾値より小さい → 生存(1)、大きい → 死亡(0)
    y_pred = (X_test["age"] < thresh).astype(int)
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)
    print(f"閾値 = {thresh:2d} 歳: 正解率 = {acc:.4f}")

# 閾値ごとの正解率を表にまとめる
age_result = pd.DataFrame({
    "threshold": thresholds,
    "accuracy": accuracies
})
age_result


次に、閾値を変えたときに正解率がどう変化するかをグラフで確認します。


In [ ]:
best_idx = np.argmax(accuracies)
best_thresh = thresholds[best_idx]
best_acc = accuracies[best_idx]

plt.figure(figsize=(9, 5))
plt.plot(thresholds, accuracies, "o-", linewidth=2, markersize=8, color="tab:purple")
plt.axvline(best_thresh, color="red", linestyle="--", label=f"最良閾値: {best_thresh}歳")
plt.scatter(best_thresh, best_acc, s=180, color="red", zorder=5)
plt.xlabel("年齢の閾値", fontsize=12)
plt.ylabel("正解率", fontsize=12)
plt.title("閾値を変えると正解率も変わる", fontsize=13)
plt.xticks(thresholds)
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


グラフの山が高いところほど、その閾値でよく分類できていることを意味します。逆に、正解率が下がっている閾値では、生存者を死亡と予測したり、死亡者を生存と予測したりする間違いが増えています。

さらに、いくつかの閾値を選んで、実際にどのように分類されているかを見てみましょう。


In [ ]:
# 代表的な閾値を選ぶ
sample_thresholds = [15, 30, 45]

fig, axes = plt.subplots(1, len(sample_thresholds), figsize=(15, 4), sharey=True)

for ax, thresh in zip(axes, sample_thresholds):
    plot_df = pd.concat([X_test, y_test], axis=1).copy()
    plot_df["pred"] = (plot_df["age"] < thresh).astype(int)
    plot_df["correct"] = plot_df["pred"] == plot_df["survived"]

    acc = accuracy_score(plot_df["survived"], plot_df["pred"])

    sns.stripplot(
        data=plot_df,
        x="age",
        y="survived",
        hue="correct",
        jitter=0.18,
        alpha=0.7,
        palette={True: "tab:green", False: "tab:red"},
        ax=ax
    )
    ax.axvline(thresh, color="black", linestyle="--", linewidth=2)
    ax.set_title(f"閾値 {thresh}歳 / 正解率 {acc:.3f}")
    ax.set_xlabel("年齢")
    ax.set_ylabel("実際のクラス")
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["死亡(0)", "生存(1)"])

handles, labels = axes[-1].get_legend_handles_labels()
legend_labels = ["正解" if label == "True" else "不正解" for label in labels]
for ax in axes:
    if ax.legend_ is not None:
        ax.legend_.remove()
fig.legend(handles, legend_labels, loc="upper center", ncol=2)
plt.tight_layout(rect=[0, 0, 1, 0.88])
plt.show()


黒い点線が閾値です。点線の左側は「生存」と予測され、右側は「死亡」と予測されます。緑の点が正解、赤の点が不正解です。閾値を動かすと、赤い点と緑の点の数が変わることが分かります。

### 2-3. 最適な閾値の特定

先ほどの正解率リストから、最も正解率が高い閾値を取り出します。


In [ ]:
best_idx = np.argmax(accuracies)
best_thresh = thresholds[best_idx]
best_acc = accuracies[best_idx]
print(f"最適な年齢閾値: {best_thresh} 歳")
print(f"その時の正解率: {best_acc:.4f}")

# 混同行列も確認
y_pred_best = (X_test["age"] < best_thresh).astype(int)
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=["死亡(0)", "生存(1)"])
disp.plot(cmap="Blues", colorbar=False)
plt.title(f"年齢閾値 = {best_thresh} 歳（正解率: {best_acc:.4f}）")
plt.tight_layout()
plt.show()


混同行列を見ると、どの種類の間違いが多いかが分かります。

- 左上：実際に死亡で、死亡と予測できた人数
- 右上：実際に死亡なのに、生存と予測した人数
- 左下：実際に生存なのに、死亡と予測した人数
- 右下：実際に生存で、生存と予測できた人数

正解率だけを見ると「良い/悪い」は分かりますが、混同行列を見ると「どのように間違えているか」まで確認できます。

### 2-4. 運賃（fare）も同様に試す

次に、運賃（fare）を使って分類してみます。タイタニックでは、運賃が高い乗客ほど上位客室に乗っていた可能性があり、生存率と関係しているかもしれません。

ここでは、

```
運賃 > 閾値 → 生存(1)
運賃 <= 閾値 → 死亡(0)
```

というルールを使います。


In [ ]:
# 運賃の閾値を 0〜100 まで 10刻みで試す
fare_thresholds = list(range(0, 101, 10))
fare_accuracies = []

for thresh in fare_thresholds:
    y_pred = (X_test["fare"] > thresh).astype(int)
    acc = accuracy_score(y_test, y_pred)
    fare_accuracies.append(acc)
    print(f"閾値 = {thresh:3d} ドル: 正解率 = {acc:.4f}")

best_fare_idx = np.argmax(fare_accuracies)
best_fare_thresh = fare_thresholds[best_fare_idx]
best_fare_acc = fare_accuracies[best_fare_idx]

plt.figure(figsize=(9, 5))
plt.plot(fare_thresholds, fare_accuracies, "s-", color="tab:orange", linewidth=2, markersize=8)
plt.axvline(best_fare_thresh, color="red", linestyle="--", label=f"最良閾値: {best_fare_thresh}ドル")
plt.scatter(best_fare_thresh, best_fare_acc, s=180, color="red", zorder=5)
plt.xlabel("運賃の閾値（ドル）", fontsize=12)
plt.ylabel("正解率", fontsize=12)
plt.title("運賃の閾値を変えたときの正解率", fontsize=13)
plt.xticks(fare_thresholds)
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"\n最適な運賃閾値: {best_fare_thresh} ドル")
print(f"その時の正解率: {best_fare_acc:.4f}")


年齢と運賃の正解率を同じグラフに並べて比較してみます。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

axes[0].plot(thresholds, accuracies, "o-", color="tab:purple", linewidth=2)
axes[0].axvline(best_thresh, color="red", linestyle="--")
axes[0].scatter(best_thresh, best_acc, s=150, color="red", zorder=5)
axes[0].set_title(f"年齢: 最良 {best_thresh}歳 / 正解率 {best_acc:.3f}")
axes[0].set_xlabel("年齢の閾値")
axes[0].set_ylabel("正解率")
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3)

axes[1].plot(fare_thresholds, fare_accuracies, "s-", color="tab:orange", linewidth=2)
axes[1].axvline(best_fare_thresh, color="red", linestyle="--")
axes[1].scatter(best_fare_thresh, best_fare_acc, s=150, color="red", zorder=5)
axes[1].set_title(f"運賃: 最良 {best_fare_thresh}ドル / 正解率 {best_fare_acc:.3f}")
axes[1].set_xlabel("運賃の閾値")
axes[1].grid(True, alpha=0.3)

plt.suptitle("特徴量と閾値によって正解率は変わる", fontsize=14)
plt.tight_layout()
plt.show()


この比較から、単純な閾値分類では「どの特徴量を使うか」と「閾値をどこに置くか」の両方が重要だと分かります。次の決定木では、このような閾値選びをコンピュータが自動で行います。

---

### 📝 演習 2-1

年齢と運賃、どちらの単一特徴量の方が高い正解率を達成できましたか？その理由をデータの特徴（年齢分布や運賃分布）から考察してください。

### 📝 演習 2-2

年齢の閾値を 1 刻みで微調整し、本当に最適な閾値を探してみてください。


In [ ]:
# ヒント：range(1, 70, 1) で細かく探索する
best_acc_fine = 0
best_thresh_fine = 0
for thresh in range(1, 70):
    y_pred = (X_test["age"] < thresh).astype(int)
    acc = accuracy_score(y_test, y_pred)
    if acc > best_acc_fine:
        best_acc_fine = acc
        best_thresh_fine = thresh
print(f"微調整後の最適閾値: {best_thresh_fine} 歳")
print(f"正解率: {best_acc_fine:.4f}")


### 📝 演習 2-3

年齢と運賃の2つを組み合わせたルール（例：「年齢 < 閾値A かつ 運賃 > 閾値B」）を作ってみてください。単一特徴量よりも精度は向上しますか？


In [ ]:
# ヒント：2つの閾値をループで探索する
best_acc_combo = 0
best_a, best_b = 0, 0
for a in range(1, 70):
    for b in range(0, 101, 5):
        y_pred = ((X_test["age"] < a) & (X_test["fare"] > b)).astype(int)
        acc = accuracy_score(y_test, y_pred)
        if acc > best_acc_combo:
            best_acc_combo = acc
            best_a, best_b = a, b
print(f"最適組み合わせ: 年齢 < {best_a} かつ 運賃 > {best_b}")
print(f"正解率: {best_acc_combo:.4f}")


---

## 3. 決定木（Decision Tree）

2章では、人間が「年齢なら何歳を閾値にするか」「運賃なら何ドルを閾値にするか」を試しながら、正解率が高くなるルールを探しました。

決定木は、この作業を**自動化した分類モデル**だと考えると理解しやすいです。つまり、決定木は次のような問いをコンピュータに任せます。

- どの特徴量で分けるのがよいか
- どの閾値で分けるのがよいか
- 1回分けたあと、次にどの特徴量・閾値で分けるのがよいか

2章の単純な閾値分類は「1つの特徴量で1回だけ分ける」方法でした。決定木は、それを**複数の特徴量で、何段階も繰り返す**方法です。

### 3-1. 決定木の仕組み

例えば2章では、次のようなルールを人間が試しました。

```
年齢 < 30 歳 → 生存
年齢 >= 30 歳 → 死亡
```

これは「年齢」という特徴量だけを使った、1段階の分類です。

決定木では、まず全ての特徴量について「どの特徴量のどの閾値で分けると、クラスが最もきれいに分かれるか」を探します。そして、分けた先のグループに対して、さらに同じことを繰り返します。

イメージとしては、以下のようなルールです。

```
┌─ age < 8 歳？
│  ├─ はい → 生存
│  └─ いいえ
│     └─ fare > 26 ドル？
│        ├─ はい → 生存
│        └─ いいえ → 死亡
```

ここで重要なのは、`age < 8` や `fare > 26` のような条件を、人間が手で決めているのではないという点です。決定木は、訓練データを見ながら、各ノードで「次にどの特徴量・どの閾値を使えば、死亡と生存がより分かれやすいか」を自動的に選びます。

この「分かれやすさ」は、scikit-learn の決定木では標準的に `gini`（ジニ不純度）という指標で評価されます。細かい式を覚える必要はありませんが、直感的には次のように考えられます。

- 死亡と生存が混ざっているグループ → 不純度が高い
- ほとんど死亡だけ、またはほとんど生存だけのグループ → 不純度が低い
- 決定木は、不純度が下がるような特徴量と閾値を選ぶ

したがって、決定木は「正解率が上がりそうな閾値を探す」という2章の考え方を、より体系的に、何度も繰り返しているモデルだと見ることができます。

### 3-2. 決定木の学習と予測

実際に決定木を学習してみます。`max_depth=3` は「最大で3段階まで質問してよい」という意味です。深さを制限すると、木が複雑になりすぎることを防げます。


In [ ]:
# 深さ3の決定木を作成
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)

# 予測
y_pred_dt = dt.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)
print(f"決定木（深さ3）の正解率: {acc_dt:.4f}")

# 混同行列
cm_dt = confusion_matrix(y_test, y_pred_dt)
plt.figure(figsize=(5, 4))
ConfusionMatrixDisplay(cm_dt, display_labels=["死亡(0)", "生存(1)"]).plot(
    cmap="Blues", colorbar=False
)
plt.title(f"決定木 (max_depth=3) — 正解率: {acc_dt:.4f}")
plt.tight_layout()
plt.show()


### 3-3. 決定木の可視化

決定木は、学習したルールを図として表示できます。これは決定木の大きな利点です。「モデルがなぜその予測をしたのか」を、人間が比較的読み取りやすいからです。


In [ ]:
plt.figure(figsize=(14, 6))
plot_tree(dt, feature_names=X.columns.tolist(),
          class_names=["死亡", "生存"],
          filled=True, rounded=True, fontsize=11)
plt.title("決定木（max_depth=3）の構造", fontsize=13)
plt.tight_layout()
plt.show()


> [!NOTE]
> 各ノードに表示される情報：
> - `gini`: ノードの不純度（小さいほど同じクラスが集中している）
> - `samples`: そのノードに含まれるデータ数
> - `value`: 各クラスのデータ数
> - `class`: そのノードの多数派クラス

図の一番上にあるノードが、最初の分割です。ここで選ばれている特徴量と閾値は、決定木が「最初に分けるならこれがよい」と判断したものです。2章で人間が年齢や運賃の閾値を試した作業を、決定木がデータから自動で行っていることを確認してください。

### 3-4. 木の深さを変えて比較

決定木は、深くすると複雑なルールを作れます。これは一見よさそうですが、訓練データに合わせすぎると、まだ見ていないテストデータでは正解率が下がることがあります。この現象を**過学習（overfitting）**と呼びます。

そこで、木の深さを変えながら、訓練データとテストデータの正解率を比較します。


In [ ]:
depths = [1, 2, 3, 5, 10, None]
train_accs = []
test_accs = []

for d in depths:
    dt_d = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt_d.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, dt_d.predict(X_train))
    test_acc = accuracy_score(y_test, dt_d.predict(X_test))
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    print(f"深さ={str(d):>4}: 訓練正解率={train_acc:.4f}  テスト正解率={test_acc:.4f}")

# グラフ化
depth_labels = [str(d) for d in depths]
x = np.arange(len(depth_labels))
plt.figure(figsize=(8, 5))
plt.plot(x, train_accs, "o-", label="訓練正解率", linewidth=2, markersize=8)
plt.plot(x, test_accs, "s-", label="テスト正解率", linewidth=2, markersize=8)
plt.xticks(x, depth_labels)
plt.xlabel("木の深さ", fontsize=12)
plt.ylabel("正解率", fontsize=12)
plt.title("決定木の深さ vs 正解率", fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


深さが浅い木は、2章の単純な閾値分類に近いシンプルなモデルです。深くなるほど、「閾値で分ける」操作を何度も繰り返せるため、より複雑な分類ができます。

ただし、訓練正解率だけが高くなり、テスト正解率が伸びない場合は、訓練データに合わせすぎている可能性があります。決定木では、`max_depth` のような設定で複雑さを調整することが大切です。

---

### 📝 演習 3-1

決定木の構造図を見て、最初の分割はどの特徴量のどの値で行われていますか？なぜその特徴量が最初に選ばれたと考えますか？

### 📝 演習 3-2

深さが `None`（制限なし）のとき、訓練正解率とテスト正解率にどのような違いがありますか？この現象を「過学習（overfitting）」の観点から説明してください。

### 📝 演習 3-3

深さ2の決定木を可視化し、その構造を人間が読めるルールとして日本語で書き出してください。


In [ ]:
dt_shallow = DecisionTreeClassifier(max_depth=2, random_state=42)
dt_shallow.fit(X_train, y_train)
plt.figure(figsize=(10, 4))
plot_tree(dt_shallow, feature_names=X.columns.tolist(),
          class_names=["死亡", "生存"],
          filled=True, rounded=True, fontsize=12)
plt.title("決定木（max_depth=2）", fontsize=13)
plt.tight_layout()
plt.show()


---

## 4. Logistic回帰（Logistic Regression）

Logistic回帰は、線形モデルを使って2クラス分類を行う最も基本的な機械学習手法の一つです。「回帰」という名前が付いていますが、目的は数値を予測することではなく、**死亡(0)か生存(1)かを分類すること**です。

2章の閾値分類は「1つの特徴量に対して閾値を置く」方法でした。3章の決定木は「特徴量と閾値の組み合わせを自動で何段階も探す」方法でした。

Logistic回帰は少し発想が違います。複数の特徴量を足し合わせて、まず1つの**線形スコア**を作ります。そのスコアを確率に変換し、「生存する確率が高いか低いか」で分類します。

### 4-1. Logistic回帰の仕組み

Logistic回帰を理解する前に、まず**線形モデル**の考え方を確認します。

線形モデルとは、複数の特徴量にそれぞれ重みをかけて、最後に足し合わせるモデルです。例えば、タイタニックのデータで「生存しやすさ」をざっくり点数化すると、次のような形になります。

```
スコア = 2.0 * sex - 0.03 * age + 0.01 * fare - 0.8 * pclass + b
```

この式では、それぞれの特徴量がスコアに足し算で効いています。

- `sex` の重みが正なら、`sex` が大きいほどスコアが上がる
- `age` の重みが負なら、年齢が高いほどスコアが下がる
- `fare` の重みが正なら、運賃が高いほどスコアが上がる
- `pclass` の重みが負なら、客室クラスの数字が大きいほどスコアが下がる

このように、特徴量を「重み付きで足し合わせる」モデルを線形モデルと呼びます。2次元で考えると、分類の境目は直線になります。3次元以上では平面や超平面になります。

Logistic回帰も、最初の部分はこの線形モデルと同じです。まず、特徴量の重み付き合計を計算します。

```
z = w1 * pclass + w2 * sex + w3 * age + w4 * sibsp + ... + b
```

この `z` は、各乗客に対する「生存しやすさのスコア」のようなものです。

- `w1, w2, w3, ...`：各特徴量の重み
- `b`：切片
- `z`：線形スコア

ここまでは、通常の線形モデルと同じです。問題は、この `z` がそのままでは確率ではないことです。`z` は `-3.2` や `5.7` のような値になり、0未満にも1より大きくもなります。

#### ロジット関数・シグモイド関数を使わない線形分類との違い

ここで少し用語を整理します。Logistic回帰では、線形スコア `z` を**ロジスティック関数（シグモイド関数）**で0〜1の確率に変換します。逆に、確率 `p` を線形スコアに戻す関数を**ロジット関数**と呼びます。

この変換を使わない単純な線形分類では、次のように分類できます。

```
if z > 0:
    予測 = 生存(1)
else:
    予測 = 死亡(0)
```

この方法でも、直線的な境界で2クラスに分けることはできます。しかし、`z` は単なるスコアなので、`z = 0.2` と `z = 8.0` の違いを「生存確率」として解釈することはできません。

一方、Logistic回帰では、線形スコア `z` をシグモイド関数に通して、0〜1の値に変換します。

```
p = 1 / (1 + exp(-z))
```

この `p` を「生存する確率」として扱います。

なぜこの式の結果が必ず0〜1の範囲に入るのでしょうか。ポイントは `exp(-z)` です。`exp(x)` は指数関数で、どんな `x` を入れても必ず正の値になります。つまり、`exp(-z)` も必ず0より大きい値です。

そのため、分母の `1 + exp(-z)` は必ず1より大きくなります。

```
exp(-z) > 0
1 + exp(-z) > 1
p = 1 / (1 + exp(-z))
```

分母が1より大きいので、`p` は必ず1より小さくなります。また、分子は1で、分母は正の値なので、`p` は必ず0より大きくなります。したがって、

```
0 < p < 1
```

となります。これにより、`p` を確率のように扱えるようになります。

具体的な値で見ると、次のようになります。

| `z` | `exp(-z)` | `p = 1 / (1 + exp(-z))` | 解釈 |
|---:|---:|---:|---|
| -5 | 約148.4 | 約0.007 | 生存確率がかなり低い |
| 0 | 1.0 | 0.5 | ちょうど境目 |
| 5 | 約0.007 | 約0.993 | 生存確率がかなり高い |

`z` が大きくなるほど `exp(-z)` は小さくなり、分母が1に近づくため、`p` は1に近づきます。逆に、`z` が小さくなるほど `exp(-z)` は大きくなり、分母が大きくなるため、`p` は0に近づきます。

```
if p > 0.5:
    予測 = 生存(1)
else:
    予測 = 死亡(0)
```

つまり、違いをまとめると以下のようになります。

| 方法 | 使う値 | 出力の意味 | 分類の仕方 |
|---|---|---|---|
| ロジット関数を使わない線形分類 | `z` | 線形スコア | `z > 0` なら生存 |
| Logistic回帰 | `p = sigmoid(z)` | 生存確率 | `p > 0.5` なら生存 |

実は、`p > 0.5` は `z > 0` と同じ境界になります。つまり、**分類境界そのものは線形**です。Logistic回帰の大きな利点は、単にクラスを予測するだけでなく、**確率として解釈できる値を出せる**ことです。

Logistic回帰は以下のように動作します。

1. 特徴量の線形結合を計算: `z = w₁×pclass + w₂×sex + w₃×age + ... + b`
2. シグモイド関数で0〜1の確率に変換: `p = 1 / (1 + exp(-z))`
3. 確率が0.5より大きければ「生存」、小さければ「死亡」と分類

線形スコア `z` とシグモイド後の確率 `p` を並べて見てみます。


In [ ]:
z = np.linspace(-10, 10, 200)
sigmoid = 1 / (1 + np.exp(-z))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ロジット関数を使わない場合：z をそのままスコアとして見る
axes[0].plot(z, z, linewidth=2.5, color="tab:gray")
axes[0].axhline(y=0, color="r", linestyle="--", label="分類境界 z=0")
axes[0].axvline(x=0, color="k", linestyle="--", alpha=0.3)
axes[0].set_xlabel("z", fontsize=12)
axes[0].set_ylabel("線形スコア z", fontsize=12)
axes[0].set_title("線形スコアのまま使う場合", fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Logistic回帰：z を 0〜1 の確率に変換する
axes[1].plot(z, sigmoid, linewidth=2.5, color="tab:blue")
axes[1].axhline(y=0.5, color="r", linestyle="--", label="分類境界 p=0.5")
axes[1].axvline(x=0, color="k", linestyle="--", alpha=0.3)
axes[1].set_xlabel("z", fontsize=12)
axes[1].set_ylabel("P(生存)", fontsize=12)
axes[1].set_title("シグモイド関数で確率に変換", fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


右のグラフでは、`z` が大きいほど生存確率が1に近づき、`z` が小さいほど0に近づきます。`z = 0` のとき確率は0.5です。そのため、Logistic回帰では `p = 0.5` を境目にして、死亡/生存を分類できます。

また、Logistic回帰では「生存」と予測するだけでなく、「生存確率は0.82」のような出力ができます。これは、分類結果の自信度を見たい場合や、閾値を0.5以外に調整したい場合に役立ちます。

### 4-2. Logistic回帰の学習と予測

scikit-learn で Logistic回帰を学習します。`predict` は0/1の分類結果を返し、`predict_proba` は各クラスの確率を返します。


In [ ]:
# Logistic回帰モデルを作成・学習
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)

# 予測
y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
print(f"Logistic回帰の正解率: {acc_lr:.4f}")

# 係数の確認
print(f"\n学習された係数:")
for name, coef in zip(X.columns, lr.coef_[0]):
    print(f"  {name} の係数: {coef:.4f}")
print(f"  切片 (b):    {lr.intercept_[0]:.4f}")

# 予測確率の確認
proba_lr = lr.predict_proba(X_test)
print("\n最初の5人の予測確率:")
for i in range(5):
    print(
        f"{i}人目: 死亡確率={proba_lr[i, 0]:.3f}, "
        f"生存確率={proba_lr[i, 1]:.3f}, "
        f"予測={y_pred_lr[i]}"
    )

# 混同行列
cm_lr = confusion_matrix(y_test, y_pred_lr)
plt.figure(figsize=(5, 4))
ConfusionMatrixDisplay(cm_lr, display_labels=["死亡(0)", "生存(1)"]).plot(
    cmap="Blues", colorbar=False
)
plt.title(f"Logistic回帰 — 正解率: {acc_lr:.4f}")
plt.tight_layout()
plt.show()


> [!NOTE]
> 係数の符号で特徴量の影響方向が分かります。正の値ならその特徴量が大きいほど「生存」の確率が上がり、負の値なら逆です。ただし、特徴量の単位が違うため、係数の絶対値をそのまま比較して「どの特徴量が一番重要」と判断するのは注意が必要です。

### 4-3. 決定境界の可視化

2次元の特徴量空間で、Logistic回帰がどこを境目に分類するかを可視化します。Logistic回帰の決定境界は、線形スコア `z = 0`、つまり生存確率 `p = 0.5` になる場所です。

ここでは見やすくするため、`age` と `fare` だけを使って別モデルを学習します。


In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay

# 決定境界は2次元で可視化するため、age と fare だけで別モデルを学習します
X_train_2d = X_train[["age", "fare"]]
y_train_2d = y_train
lr_2d = LogisticRegression(random_state=42, max_iter=1000)
lr_2d.fit(X_train_2d, y_train_2d)

DecisionBoundaryDisplay.from_estimator(
    lr_2d, X_train_2d,
    response_method="predict",
    cmap="Pastel1",
    alpha=0.6
)
# 訓練データを重ねて表示
scatter = plt.scatter(X_train_2d["age"], X_train_2d["fare"],
                      c=y_train, cmap="Pastel1", alpha=0.5, edgecolors="gray", s=30)
plt.xlabel("age", fontsize=12)
plt.ylabel("fare", fontsize=12)
plt.title("Logistic回帰の決定境界", fontsize=13)
plt.tight_layout()
plt.show()


この図で色が切り替わる境目が、Logistic回帰の決定境界です。決定木のように階段状に細かく分けるのではなく、基本的には1本の直線で分類します。そのため、関係が単純なときは扱いやすい一方、複雑な境界は表現しにくいという特徴があります。

### 4-4. 3モデルの比較


In [ ]:
# 閾値分類（年齢）
y_pred_thresh = (X_test["age"] < best_thresh).astype(int)
acc_thresh = accuracy_score(y_test, y_pred_thresh)

# 結果をまとめ表示
results = {
    "単純閾値 (age)": acc_thresh,
    "決定木 (depth=3)": acc_dt,
    "Logistic回帰": acc_lr
}

for name, acc in results.items():
    print(f"{name}: {acc:.4f}")

# 棒グラフ
plt.figure(figsize=(7, 5))
bars = plt.bar(results.keys(), results.values(), color=["#66c2a5", "#fc8d62", "#8da0cb"], width=0.5)
plt.ylabel("正解率", fontsize=12)
plt.title("3モデルの比較", fontsize=13)
plt.ylim(0.75, 1.0)
for bar, val in zip(bars, results.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f"{val:.4f}", ha="center", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


### 4-4.4. その他の分類モデルへの差し替え

session5-6 では、Logistic回帰以外にも複数の分類モデルを紹介しました。scikit-learn の便利な点は、多くの分類モデルがほぼ同じ形で使えることです。

Logistic回帰では、次の3ステップで学習と予測を行いました。


In [ ]:
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


他の分類モデルを使う場合、多くの場合は**1行目のモデル名を変更するだけ**です。`fit` と `predict` の使い方はほとんど同じです。

| モデル | 変更する部分 | 特徴 |
|---|---|---|
| Logistic回帰 | `LogisticRegression(...)` | 直線的な境界。係数と確率を解釈しやすい |
| 決定木 | `DecisionTreeClassifier(...)` | 「もし〜なら」のルールとして説明しやすい |
| Random Forest | `RandomForestClassifier(...)` | 多数の決定木の多数決。安定しやすい |
| Gradient Boosting | `GradientBoostingClassifier(...)` | 弱い木を順番に改善していく。高精度になりやすい |
| K近傍法 | `KNeighborsClassifier(...)` | 近いデータの多数決で分類する |
| SVM | `SVC(...)` | 境界からの余裕（マージン）を大きくするように分類する |
| Naive Bayes | `GaussianNB()` | 確率分布の仮定に基づいて高速に分類する |
| Neural Net | `MLPClassifier(...)` | 多層の重み付き計算で複雑な関係を表現する |

例えば、Logistic回帰を Random Forest に変更する場合は、以下のようにします。


In [ ]:
# Logistic回帰
model = LogisticRegression(random_state=42, max_iter=1000)

# Random Forest に変更
model = RandomForestClassifier(n_estimators=100, random_state=42)


その後の学習・予測・評価は同じです。


In [ ]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("正解率:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["死亡(0)", "生存(1)"]))


このように、scikit-learn ではモデルを差し替えながら、同じ評価方法で比較できます。次の演習では、1つずつモデルを入れ替える方法から始め、最後に `for` 文でまとめて比較します。

---

### 📝 演習 4-1

Logistic回帰で学習された係数を見て、各特徴量が生存確率に与える影響の向きを確認してください。なお、特徴量は単位やスケールが違うため、係数の絶対値をそのまま比較して「どちらが重要」と判断するのは慎重に行う必要があります。総合課題ステップ6のように正規化した後の係数も確認してみましょう。

### 📝 演習 4-2

`other_models` の中から Logistic回帰以外のモデルを1つ選び、Logistic回帰と同じように正解率・`classification_report`・混同行列を表示してください。

**ヒント：** まずは `selected_model = ...` の1行だけを入れ替えます。`fit`、`predict`、評価の書き方は同じです。


In [ ]:
# 例：Random Forest を使う場合
selected_model = RandomForestClassifier(n_estimators=100, random_state=42)
selected_model.fit(X_train, y_train)
y_pred_selected = selected_model.predict(X_test)

print("正解率:", accuracy_score(y_test, y_pred_selected))
print(classification_report(y_test, y_pred_selected, target_names=["死亡(0)", "生存(1)"]))

cm_selected = confusion_matrix(y_test, y_pred_selected)
ConfusionMatrixDisplay(cm_selected, display_labels=["死亡(0)", "生存(1)"]).plot(
    cmap="Blues",
    colorbar=False
)
plt.title("選んだモデルの混同行列")
plt.tight_layout()
plt.show()


次に、例えば以下のようにモデルだけを変更して、成績がどう変わるか確認してください。


In [ ]:
# 決定木
selected_model = DecisionTreeClassifier(max_depth=3, random_state=42)

# K近傍法
selected_model = KNeighborsClassifier(n_neighbors=5)

# Naive Bayes
selected_model = GaussianNB()


### 📝 演習 4-3（forで全モデルを比較）

1つずつモデルを入れ替える代わりに、`for` 文を使ってすべてのモデルの成績をまとめて求めてください。

**ヒント：** 以下の `other_models` を使います。辞書のキーがモデル名、値が実際のモデルです。


In [ ]:
other_models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "SVM (RBF)": SVC(kernel="rbf"),
    "Naive Bayes": GaussianNB(),
    "Neural Net": MLPClassifier(hidden_layer_sizes=(50, 50), max_iter=1000, random_state=42),
}


まずは、正解率だけを表示してみましょう。


In [ ]:
for name, model in other_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name}: {acc:.4f}")


余裕があれば、以下のコードを参考にして、正解率・生存を見つける力・死亡を見つける力を表にまとめてください。


In [ ]:
model_results = []

for name, model in other_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    model_results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "生存を見つける力": tp / (tp + fn),
        "死亡を見つける力": tn / (tn + fp),
    })

pd.DataFrame(model_results).sort_values("accuracy", ascending=False)


### 📝 演習 4-4（分類曲線・分類境界の比較）

`age` と `fare` の2つだけを使って、モデルごとの分類境界を描いてください。分類境界を見ると、モデルがどのような形で死亡/生存を分けているかが分かります。

**ヒント：** 決定境界を2次元で描くため、ここでは `X_train[["age", "fare"]]` だけを使います。


In [ ]:
X_train_2d = X_train[["age", "fare"]]
y_train_2d = y_train

boundary_models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.ravel()

for ax, (name, model) in zip(axes, boundary_models.items()):
    model.fit(X_train_2d, y_train_2d)

    DecisionBoundaryDisplay.from_estimator(
        model,
        X_train_2d,
        response_method="predict",
        cmap="Pastel1",
        alpha=0.6,
        ax=ax,
    )

    ax.scatter(
        X_train_2d["age"],
        X_train_2d["fare"],
        c=y_train_2d,
        cmap="Pastel1",
        edgecolors="gray",
        s=25,
        alpha=0.6,
    )
    ax.set_title(name)
    ax.set_xlabel("age")
    ax.set_ylabel("fare")

plt.tight_layout()
plt.show()


分類境界を見て、以下を考えてください。

1. 正解率が最も高いモデルはどれですか？
2. 分類境界が一番シンプルなモデルはどれですか？
3. 分類境界が一番複雑なモデルはどれですか？
4. 複雑な境界のモデルは、必ず良いモデルと言えるでしょうか？過学習の観点から考えてください。

---

## 🏆 本日の総合課題

以下のステップをすべて実行し、考察をコードのコメントとして記録してください。


In [ ]:
# ステップ1：データ読み込みから前処理までを一通り実行する
# （欠損値補完・特徴量選択・train_test_split）


In [ ]:
# ステップ2：単純な閾値分類で年齢と運賃の最適な閾値を探す
# どちらの方が精度が高いか比較する


In [ ]:
# ステップ3：決定木（深さ2〜5）を学習し、構造を可視化する
# どの特徴量が最初に分割されるか確認する


In [ ]:
# ステップ4：Logistic回帰を学習し、係数を確認する
# 決定境界を可視化する


In [ ]:
# ステップ5：3モデルを比較し、以下をコメントで書く（各項目2〜3文）
# - どのモデルが最も高精度か
# - 単純閾値が「解釈容易」である利点を説明する
# - 決定木が「非線形な関係」を扱える利点を説明する
# - Logistic回帰が「確率出力」を提供する利点を説明する


In [ ]:
# ステップ6（応用）：正規化の影響を確認する
# Logistic回帰は特徴量のスケールに敏感。StandardScalerで正規化してから
# 再学習し、係数と正解率がどう変わるか比較する
#
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)
# lr_scaled = LogisticRegression(random_state=42, max_iter=1000)
# lr_scaled.fit(X_train_scaled, y_train)
# print("正規化後の係数:", lr_scaled.coef_)
# print("正解率:", accuracy_score(y_test, lr_scaled.predict(X_test_scaled)))
